In [1]:
#!pip install gdown

In [2]:
#!gdown https://drive.google.com/uc?id=1MhctnTnP8PnBYKdlAtY7y_GVqXZzV2Jz

In [3]:
import anndata as ad
import pandas as pd

In [4]:
pd.options.display.max_columns = 100

## Check L1000 structure

In [ ]:
adata_l1000 = ad.read_h5ad('./l1000_standardized.h5ad')

In [ ]:
adata_sci = ad.read_h5ad('./data/sciplex/pseudobulk/subsample/srivatsan20_sciplex3_subsample.h5ad')

In [ ]:
adata_l1000

In [ ]:
adata_sci

### .obs

In [ ]:
# Excessive columns in comparison to our reference structure in adata_sci:
adata_l1000.obs[list(set(adata_l1000.obs.columns) - set(adata_sci.obs.columns))]

In [ ]:
# No columns in comparison to our reference structure in adata_sci:
adata_sci.obs[list(set(adata_sci.obs.columns) - set(adata_l1000.obs.columns))]

### .var

In [ ]:
# Excessive columns in comparison to our reference structure in adata_sci:
adata_l1000.var[list(set(adata_l1000.var.columns) - set(adata_sci.var.columns))]

In [ ]:
# No columns in comparison to our reference structure in adata_sci:
adata_sci.var[list(set(adata_sci.var.columns) - set(adata_l1000.var.columns))]

### .X

In [ ]:
adata_l1000.X

In [ ]:
adata_sci.X

### .layers

In [ ]:
adata_l1000.layers

In [ ]:
adata_sci.layers

## Content

In [ ]:
adata_sci.obs.dtypes

In [ ]:
adata_l1000.obs[list(set(adata_l1000.obs.columns).intersection(set(adata_sci.obs.columns)))].dtypes

In [ ]:
adata_l1000.var.dtypes

In [ ]:
adata_l1000.obs[['pert_dose_unit']].value_counts(dropna=False)

In [ ]:
adata_l1000.obs[['pert_type']].value_counts(dropna=False)

## Summary:

#### About the general structure:

Standardized l1000 is an anndata object containing obs, var annotations and not sparse count matrix .X. The dictionary-like object layers is empty (I think it could be as l1000 does not have psbulk_props information).

#### Dataset size:
I see that the n_var equals `978`, it is related to the landmark genes, but there are also inferred genes according to the [cmapBQ documentation](https://cmapbq.readthedocs.io/en/latest/cmapBQ.html#module-cmapBQ.query):
+ Best-inferred set of `10,174` genes
+ All inferred genes including `12,328` genes

Are we interested in only landmark genes?

#### Structure of annotations:
+ **.obs** has excessive columns:
  
  `['rna_well', 'pert_itime', 'cmap_name', 'time_val', 'inv_level_10',
       'nearest_dose', 'pert_time_bin', 'project_code', 'perturbation_label',
       'compound_aliases', 'cell_mfc_name', 'dose_bin_uM', 'mw_g_per_mol',
       'pert_id', 'pert_time_unit', 'qc_iqr', 'qc_pass', 'rna_plate',
       'qc_slope', 'sample_id', 'pert_time', 'failure_mode', 'time_unit',
       'inchi_key', 'count_cv', 'qc_f_logp', 'pert_dose_unit', 'count_mean',
       'bead_batch', 'pert_dose', 'dyn_range', 'pert_mfc_id', 'dose_val',
       'build_name', 'alternative_alias', 'pert_idose']`
<br>
  and also lacks of some columns from the standard schema:

  `['psbulk_counts', 'dataset', 'sex', 'library', 'self_reported_ethnicity',
  'tissue', 'guide', 'assay', 'disease', 'stimulation', 'tissue_type',
  'organism', 'suspension_type', 'development_stage']`

  additionally there is a difference in indices of .obs dataframes, in the original schema there is `sample_id` which is constructed as `plate + '_' + well + '_' + perturbagen + '_' + cell_type`, l1000 .obs indices are integers.

+ **.var** has excessive columns:

  `['pr_is_lm', 'pr_gene_symbol']`

    additionally there is a difference in indices of .var dataframes, in the original schema there is `ensembl_id` as an index, and l1000 .var index is `pr_gene_symbol` which corresponds to the gene symbols.

+ **.X** is non-sparse `np.array` which contains `float` values, I think it is reasonable as l1000 does not contain the pure counts.

+ **.layers** is empty, it is reasonable as it might contains just additional information (e.g. produced by `.pp.pseudobulk` function from the `decoupler==2.1.0`).



#### Content of l1000 annotations (based on the original schema):

+ **.obs**

`plate <category>` - ok, id of the plate (e.g. `ABY001_A375_XH_X1_B15`)

`well <category>` - ok, id of the plate (e.g. `D10`)

`cell_type <category>` - sould be **Cellosaurus**, currently `A375`.

`perturbagen <category>` - id of perturbagen (e.g. `943`?, `BRD-A61304759`, `BFP_R1`); we have a name of perturbagen in the original schema.

`pert_type <category>` - indicates the type of perturbagen (e.g. `ctl_vector`, `trt_cp`, `ctl_x`, `ctl_vehicle`, `ctl_untrt`). The original annotation has (e.g. `compound`, `genetic`...). Also, do we consider only compounds and not genetic? In the case we have compounds we need to merge `ctl_vehicle` and `trt_cp` type to `compound` and write (is_contros = True). Here the control compound perturbagens: `['DMSO', 'H2O', 'PBS', 'LIPOFECTAMINE', 'POLYBRENE']`.

`is_control <bool>` - ok, indicates if the perturbation is control or not. (e.g. currently, in l1000 if the perturbagen is in `['DMSO', 'H2O', 'PBS', 'LIPOFECTAMINE', 'POLYBRENE']` - then perturbation is control)

`pert_dose_uM <float64>` - If we consider only compounds as perturbagens then we also need to be able to convert `'ug/ml', '%'` to our `pert_dose_uM`. Currently it is empty. Additionally, I have no idea how to determine uM for `ctl_vector`, `ctl_x`, `ctl_untrt`.

`pert_time_h <float64>` - ok, but I do not know how to convert the info -666.0	into hours.

`suspension_type <category>` - need to be added.

`tissue <category>` - need to be added. Suppose could be taken from `cell_lineage` of `cellinfo` file.


`tissue_type <category>` - need be added.

`disease <category>` - I suppose could be combined as columns `primary_disease` and `subtype` from `cellinfo` file.

`library <category>` - need be added (suppose it should be a column with NaN values).

`stimulation <category>` - need be added (could be a column with NaN values, but I am not sure, need to take a look close).

`guide <category>` - need be added (if we consider compounds I suppose it should be a column with NaN values).

`dataset <category>` - need be added: a name of the dataset, specified manually.

`assay <category>` - need be added (suppose, it could be `L1000 mRNA profiling assay`, but i need to check with the correct efo version).

`development_stage <category>` - need to be added (suppose, it could be derived from `donor_age` column of `cellinfo` file).

`organism <category>` - need to be added (all of cells are from `human` tissues?).

`sex <category>` - need to be added (suppose, it could be derived from `donor_sex` column of `cellinfo` file).

`self_reported_ethnicity <category>` - need to be added (suppose, it could be derived from `donor_ethnicity` column of `cellinfo` file).

`pubchem_cid <category>` - ok. (I have to check the column)

`psbulk_cells <int64>` - it exists, but I am not sure that the value 1 is correct; but also do not know how to derive the correct values. Maybe we need to insert Nans.

`psbulk_counts <int64>` - need to be added: NaN values of the sum of all counts from .X (maybe the second option does not make sense)?

+ **.var**

`symbol <category>` - suppose `pr_gene_symbol` should be renamed to `symbol` and converted to the `<category>`

`ensembl_id` should be added as an index instead of `pr_gene_symbol`.